# Classifier Model Training — Final

Trains the final LightGBM classifier on `dataset/FINAL_DATASET.parquet` (output of `feature_engineering_final.ipynb`).

**Input:** `dataset/FINAL_DATASET.parquet` — 61 features, 18.2M rows

**Output:**
- `models/lgbm_delay_classifier_final.pkl` — trained model
- `models/feature_list_final.txt` — 61 feature names
- `models/model_final_metadata.json` — AUC, params, metadata

**Results:** AUC 0.8629 | Precision 71.9% | Recall 61.6% | F1 0.663 @ threshold 0.65

In [2]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.metrics import roc_auc_score
import joblib
import json
import time
import os
import gc

flights = pd.read_parquet('dataset/FINAL_DATASET.parquet')
flights['FL_DATE'] = pd.to_datetime(flights['FL_DATE'])
print(f'Loaded: {flights.shape[0]:,} rows x {flights.shape[1]} cols')

Loaded: 18,227,796 rows x 120 cols


## Feature Definitions

In [3]:
ALL_FEATURES = [
    'MONTH', 'DAY_OF_WEEK', 'DEP_HOUR', 'ARR_HOUR', 'IS_HOLIDAY',
    'DISTANCE', 'profit_margin', 'origin_monthly_passengers',
    'origin_temp', 'origin_dew_point', 'origin_pressure',
    'origin_wind_dir', 'origin_wind_speed', 'origin_precip_1hr',
    'origin_weather_severity',
    'dest_temp', 'dest_dew_point', 'dest_pressure',
    'dest_wind_dir', 'dest_wind_speed', 'dest_precip_1hr',
    'dest_weather_severity',
    'airline_delay_rate_30d', 'origin_delay_rate_30d', 'dest_delay_rate_30d',
    'route_delay_rate_30d', 'origin_avg_taxi_out_30d',
    'flight_num_delay_rate_30d', 'origin_hour_delay_rate_30d',
    'carrier_origin_delay_rate_30d', 'dest_hour_delay_rate_30d',
    'airline_delay_rate_7d', 'origin_delay_rate_7d',
    'cascade_score', 'cascade_delay_minutes', 'hours_since_last_delay',
    'hourly_flight_count', 'scheduled_turnaround_buffer', 'tail_flight_num_today',
    'dest_hourly_flight_count',
    'inbound_arr_delay_3h', 'dest_inbound_arr_delay_3h',
    'prev_tail_arr_delay', 'national_hub_delay_2h',
    'OP_UNIQUE_CARRIER', 'ORIGIN', 'DEST', 'airline_cluster_label',
    'origin_pressure_delta_3h', 'dest_pressure_delta_3h',
    'origin_wind_speed_delta_3h', 'dest_wind_speed_delta_3h',
    'day_of_year',
    'origin_dep_delay_rate_1h', 'dest_dep_delay_rate_1h',
    'origin_stress_index', 'real_time_turn_gap',
    'tail_delays_today', 'tail_active_hours',
    'origin_pressure_drop_stress', 'airport_fatigue_index',
]

CAT_FEATURES = ['OP_UNIQUE_CARRIER', 'ORIGIN', 'DEST', 'airline_cluster_label']

assert len(ALL_FEATURES) == 61, f'Expected 61 features, got {len(ALL_FEATURES)}'
missing = [f for f in ALL_FEATURES if f not in flights.columns]
assert len(missing) == 0, f'Missing features: {missing}'
print(f'Features: {len(ALL_FEATURES)}')

Features: 61


## Train/Test Split

Temporal split — no random shuffle. Train: Jan 2023 – Dec 2024 | Test: Jan 2025 – Aug 2025

In [4]:
for col in CAT_FEATURES:
    flights[col] = flights[col].astype('category')

train = flights[flights['FL_DATE'] < '2025-01-01']
test  = flights[flights['FL_DATE'] >= '2025-01-01']

X_train = train[ALL_FEATURES]
y_train = train['ARR_DEL15']
X_test  = test[ALL_FEATURES]
y_test  = test['ARR_DEL15']

print(f'Train: {X_train.shape[0]:,} | Test: {X_test.shape[0]:,}')
print(f'Train delay rate: {y_train.mean():.3f} | Test delay rate: {y_test.mean():.3f}')

del train, test
gc.collect()

Train: 13,708,670 | Test: 4,519,126
Train delay rate: 0.207 | Test delay rate: 0.230


0

## Model Training

Optuna-tuned LightGBM classifier. Parameters from 50-trial TPE search on 2M subsample.

In [5]:
model_final = lgb.LGBMClassifier(
    objective='binary', metric='auc', is_unbalance=True,
    verbose=-1, random_state=42, n_jobs=-1,
    n_estimators=10000,
    learning_rate=0.009811877112556893,
    num_leaves=312,
    max_depth=13,
    min_child_samples=278,
    subsample=0.722267800117643,
    colsample_bytree=0.5308228457271158,
    reg_alpha=0.16398261731969382,
    reg_lambda=3.406904427387338,
    min_split_gain=0.9742015244622154,
    max_bin=200,
)

start = time.time()
model_final.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    callbacks=[lgb.early_stopping(200), lgb.log_evaluation(200)],
)
elapsed = time.time() - start

y_pred_final = model_final.predict_proba(X_test)[:, 1]
auc_final = roc_auc_score(y_test, y_pred_final)

print(f"\n{'=' * 60}")
print(f'FINAL MODEL — {len(ALL_FEATURES)} Features')
print(f'  AUC:          {auc_final:.4f}')
print(f'  59-feature:   0.8623')
print(f'  Optuna 48:    0.8578')
print(f'  Baseline 48:  0.8572')
print(f'  Iteration:    {model_final.best_iteration_}')
print(f'  Time:         {elapsed/60:.1f} min')
print(f"{'=' * 60}")

Training until validation scores don't improve for 200 rounds
[200]	valid_0's auc: 0.85025
[400]	valid_0's auc: 0.854642
[600]	valid_0's auc: 0.857006
[800]	valid_0's auc: 0.858502
[1000]	valid_0's auc: 0.85948
[1200]	valid_0's auc: 0.860201
[1400]	valid_0's auc: 0.860753
[1600]	valid_0's auc: 0.861162
[1800]	valid_0's auc: 0.861479
[2000]	valid_0's auc: 0.861745
[2200]	valid_0's auc: 0.861955
[2400]	valid_0's auc: 0.862133
[2600]	valid_0's auc: 0.862269
[2800]	valid_0's auc: 0.862387
[3000]	valid_0's auc: 0.862471
[3200]	valid_0's auc: 0.862537
[3400]	valid_0's auc: 0.862598
[3600]	valid_0's auc: 0.862636
[3800]	valid_0's auc: 0.862678
[4000]	valid_0's auc: 0.86271
[4200]	valid_0's auc: 0.862729
[4400]	valid_0's auc: 0.862756
[4600]	valid_0's auc: 0.862773
[4800]	valid_0's auc: 0.862786
[5000]	valid_0's auc: 0.86281
[5200]	valid_0's auc: 0.862823
[5400]	valid_0's auc: 0.862835
[5600]	valid_0's auc: 0.862844
[5800]	valid_0's auc: 0.862851
[6000]	valid_0's auc: 0.862855
[6200]	valid_0's

## Feature Importance

In [6]:
imp = pd.DataFrame({
    'feature': ALL_FEATURES,
    'importance': model_final.feature_importances_
}).sort_values('importance', ascending=False)

print('Full feature rankings:')
for rank, (_, row) in enumerate(imp.iterrows(), 1):
    print(f"  #{rank:<3} {row['feature']:<40} {row['importance']:>10,}")

Full feature rankings:
  #1   ORIGIN                                      297,467
  #2   DEST                                        278,808
  #3   OP_UNIQUE_CARRIER                            77,674
  #4   day_of_year                                  67,607
  #5   origin_pressure                              51,869
  #6   dest_inbound_arr_delay_3h                    48,358
  #7   inbound_arr_delay_3h                         48,295
  #8   national_hub_delay_2h                        48,244
  #9   dest_pressure                                44,446
  #10  origin_delay_rate_7d                         42,231
  #11  real_time_turn_gap                           40,867
  #12  scheduled_turnaround_buffer                  39,717
  #13  origin_delay_rate_30d                        39,430
  #14  airline_delay_rate_7d                        37,874
  #15  origin_dew_point                             37,135
  #16  prev_tail_arr_delay                          36,839
  #17  airline_delay_rate_30d    

## Threshold Analysis

In [7]:
print(f"{'Threshold':<10} {'Precision':<12} {'Recall':<12} {'F1':<12} {'Flagged%':<10}")
print('-' * 55)
for t in [0.45, 0.50, 0.55, 0.60, 0.65, 0.70]:
    y_bin = (y_pred_final >= t).astype(int)
    tp = ((y_bin == 1) & (y_test == 1)).sum()
    fp = ((y_bin == 1) & (y_test == 0)).sum()
    fn = ((y_bin == 0) & (y_test == 1)).sum()
    prec = tp / (tp + fp) if (tp + fp) > 0 else 0
    rec  = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1   = 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0
    flagged = y_bin.mean() * 100
    print(f'  {t:.2f}     {prec:.4f}       {rec:.4f}       {f1:.4f}       {flagged:.1f}%')

Threshold  Precision    Recall       F1           Flagged%  
-------------------------------------------------------
  0.45     0.5284       0.7632       0.6244       33.2%
  0.50     0.5770       0.7252       0.6427       28.9%
  0.55     0.6253       0.6881       0.6552       25.3%
  0.60     0.6725       0.6517       0.6619       22.3%
  0.65     0.7189       0.6156       0.6633       19.7%
  0.70     0.7641       0.5793       0.6590       17.4%


## Save Model + Artifacts

In [9]:
# Save model
joblib.dump(model_final, 'models/lgbm_delay_classifier_final.pkl')

# Save feature list
with open('models/feature_list_final.txt', 'w') as f:
    for feat in ALL_FEATURES:
        f.write(feat + '\n')

# Save metadata
metadata = {
    'auc': float(auc_final),
    'features': len(ALL_FEATURES),
    'best_iteration': int(model_final.best_iteration_),
    'train_rows': int(X_train.shape[0]),
    'test_rows': int(X_test.shape[0]),
    'categorical_features': CAT_FEATURES,
    'params': {
        'learning_rate': 0.009811877112556893,
        'num_leaves': 312,
        'max_depth': 13,
        'min_child_samples': 278,
        'subsample': 0.722267800117643,
        'colsample_bytree': 0.5308228457271158,
        'reg_alpha': 0.16398261731969382,
        'reg_lambda': 3.406904427387338,
        'min_split_gain': 0.9742015244622154,
        'max_bin': 200,
    }
}

with open('models/model_final_metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)

size_mb = os.path.getsize('models/lgbm_delay_classifier_final.pkl') / 1024 / 1024
print(f'✓ Model saved: models/lgbm_delay_classifier_final.pkl ({size_mb:.1f} MB)')
print(f'✓ Features saved: models/feature_list_final.txt ({len(ALL_FEATURES)} features)')
print(f'✓ Metadata saved: models/model_final_metadata.json')
print(f'\nFinal AUC: {auc_final:.4f} | Features: {len(ALL_FEATURES)} | Iterations: {model_final.best_iteration_}')

✓ Model saved: models/lgbm_delay_classifier_final.pkl (277.6 MB)
✓ Features saved: models/feature_list_final.txt (61 features)
✓ Metadata saved: models/model_final_metadata.json

Final AUC: 0.8629 | Features: 61 | Iterations: 7211
